In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [2]:
# Loading data
df = pd.read_csv(r'C:\Users\USER\Downloads\laptops_prices.csv', encoding='utf-8-sig')  # Fixes BOM issues
print(df.columns.tolist())

['Manufacturer', 'Model Name', 'Category', 'Screen Size', 'Screen', 'CPU', 'RAM', ' Storage', 'GPU', 'Operating System', 'Operating System Version', 'Weight', 'Price']


In [3]:
# Encoding of some columns with non-homogeneous data structure

# From CPU column
df['CPU_Brand'] = df['CPU'].str.split().str[0]  # Intel/AMD
df['CPU_Speed'] = df['CPU'].str.extract(r'(\d+\.?\d*)GHz').astype(float)

# From GPU column
df['GPU_Brand'] = df['GPU'].str.split().str[0]  # Intel/NVIDIA/AMD


In [4]:
# Extract numbers from RAM column (e.g., '4GB' → 4)
df['RAM'] = df['RAM'].str.extract('(\d+)').astype(float)

# Verify conversion
print(df['RAM'].head())

0     6.0
1    16.0
2    12.0
3     4.0
4     6.0
Name: RAM, dtype: float64


In [5]:
# Remove leading/trailing spaces from ALL column names
df.columns = df.columns.str.strip()


In [6]:
def clean_storage_simple(value):
    value = str(value).strip()
    
    # Extract first number found in the string
    import re
    match = re.search(r'(\d+)', value)
    if match:
        num = float(match.group(1))
        if 'TB' in value.upper():  # Case-insensitive check
            return num * 1000
        return num
    return float('nan')

df['Storage'] = df['Storage'].apply(clean_storage_simple)

# Check if we got better results
print("After simple cleaning:")
print(df['Storage'].head(10))

After simple cleaning:
0      1000.0
1    256000.0
2       512.0
3       128.0
4       256.0
5       256.0
6       500.0
7       500.0
8      1000.0
9    128000.0
Name: Storage, dtype: float64


In [7]:
print("Raw Storage values:")
print(df['Storage'].head(10).to_list())  # First 10 values
print("\nUnique Storage values:", df['Storage'].unique())

Raw Storage values:
[1000.0, 256000.0, 512.0, 128.0, 256.0, 256.0, 500.0, 500.0, 1000.0, 128000.0]

Unique Storage values: [1.00e+03 2.56e+05 5.12e+02 1.28e+02 2.56e+02 5.00e+02 1.28e+05 3.20e+01
 1.60e+01 2.00e+03 5.12e+05 6.40e+01 1.00e+00]


In [8]:
# Setting training parameters
X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
# Clean both X_train and X_test
for data in [X_train, X_test]:
    data['RAM'] = data['RAM'].astype(str).str.extract('(\d+)').astype(float)

In [10]:
# Convert RAM column to string first (safely handles any input type)
df['RAM'] = df['RAM'].astype(str)

# Now extract numeric part and convert to float
df['RAM'] = (
    df['RAM']
    .str.extract(r'(\d+)', expand=False)  # Extract first number found
    .astype(float)
)

# Handle any remaining missing values
if df['RAM'].isna().any():
    print(f"Warning: {df['RAM'].isna().sum()} missing values in RAM")
    df['RAM'] = df['RAM'].fillna(df['RAM'].median())

# Verify
print("RAM after cleaning:")
print(df['RAM'].head())
print(f"Data type: {df['RAM'].dtype}")
print(f"Unique values: {df['RAM'].unique()}")

RAM after cleaning:
0     6.0
1    16.0
2    12.0
3     4.0
4     6.0
Name: RAM, dtype: float64
Data type: float64
Unique values: [ 6. 16. 12.  4.  8.  2. 64. 32.]


In [11]:
# Assuming df is your cleaned DataFrame

numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)


Numerical features: ['RAM', 'Storage', 'Price', 'CPU_Speed']
Categorical features: ['Manufacturer', 'Model Name', 'Category', 'Screen Size', 'Screen', 'CPU', 'GPU', 'Operating System', 'Operating System Version', 'Weight', 'CPU_Brand', 'GPU_Brand']


In [12]:
# 1. Clean ALL data first
df['RAM'] = df['RAM'].astype(str).str.extract('(\d+)').astype(float)

# 2. THEN split into train/test
X = df.drop('Price', axis=1)
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Verify training data
print("X_train RAM sample:", X_train['RAM'].head())
print("X_train RAM dtype:", X_train['RAM'].dtype)

X_train RAM sample: 172     4.0
183    16.0
17      8.0
24      4.0
132     8.0
Name: RAM, dtype: float64
X_train RAM dtype: float64


In [13]:
# 7. Update features
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['RAM', 'Storage', 'Price', 'CPU_Speed']
Categorical features: ['Manufacturer', 'Model Name', 'Category', 'Screen Size', 'Screen', 'CPU', 'GPU', 'Operating System', 'Operating System Version', 'Weight', 'CPU_Brand', 'GPU_Brand']


In [14]:
import pandas as pd
import re
import numpy as np

# 1. First inspect your Screen column
print("Original Screen values sample:\n", df['Screen'].head())

# 2. Robust extraction function
def extract_resolution(res_str):
    try:
        # Handle multiple formats: "1366x768", "1920 x 1080", "Full HD 1920x1080"
        nums = re.findall(r'\d+', str(res_str))
        if len(nums) >= 2:
            return float(nums[0]), float(nums[1])
        return np.nan, np.nan
    except:
        return np.nan, np.nan

# 3. Apply extraction
df['Screen_Width'], df['Screen_Height'] = zip(*df['Screen'].apply(extract_resolution))

# 4. Verify
print("\nExtraction results:")
print(df[['Screen', 'Screen_Width', 'Screen_Height']].head(10))

# 5. Handle missing values
missing = df['Screen_Width'].isna().sum()
if missing > 0:
    print(f"\nFound {missing} missing resolutions. Filling with medians.")
    df['Screen_Width'] = df['Screen_Width'].fillna(df['Screen_Width'].median())
    df['Screen_Height'] = df['Screen_Height'].fillna(df['Screen_Height'].median())

# 6. Final check
print("\nFinal verification:")
print("Screen_Width stats:")
print(df['Screen_Width'].describe())
print("\nScreen_Height stats:")
print(df['Screen_Height'].describe())



Original Screen values sample:
 0                                     1366x768
1                            Full HD 1920x1080
2    IPS Panel Full HD / Touchscreen 1920x1080
3                            Full HD 1920x1080
4                            Full HD 1920x1080
Name: Screen, dtype: object

Extraction results:
                                      Screen  Screen_Width  Screen_Height
0                                   1366x768        1366.0          768.0
1                          Full HD 1920x1080        1920.0         1080.0
2  IPS Panel Full HD / Touchscreen 1920x1080        1920.0         1080.0
3                          Full HD 1920x1080        1920.0         1080.0
4                          Full HD 1920x1080        1920.0         1080.0
5                                   1366x768        1366.0          768.0
6                                   1366x768        1366.0          768.0
7                                   1366x768        1366.0          768.0
8                 

In [16]:
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['RAM', 'Storage', 'Price', 'CPU_Speed']
Categorical features: ['Manufacturer', 'Model Name', 'Category', 'Screen Size', 'Screen', 'CPU', 'GPU', 'Operating System', 'Operating System Version', 'Weight', 'CPU_Brand', 'GPU_Brand']


In [17]:
print("Columns in X_train:", X_train.columns.tolist())

Columns in X_train: ['Manufacturer', 'Model Name', 'Category', 'Screen Size', 'Screen', 'CPU', 'RAM', 'Storage', 'GPU', 'Operating System', 'Operating System Version', 'Weight', 'CPU_Brand', 'CPU_Speed', 'GPU_Brand']


In [19]:
print("\nFinal verification:")
print("Screen_Width range:", df['Screen_Width'].min(), "-", df['Screen_Width'].max())
print("Screen_Height range:", df['Screen_Height'].min(), "-", df['Screen_Height'].max())
print("Missing values:", df[numerical_features].isna().sum())


Final verification:
Screen_Width range: 4.0 - 3200.0
Screen_Height range: 768.0 - 3840.0
Missing values: RAM          0
Storage      0
Price        0
CPU_Speed    0
dtype: int64


In [20]:
# 1. First ensure ALL feature engineering is done BEFORE the train/test split
def create_features(df):
    """Create all derived features in one place"""
    # Extract screen dimensions
    def extract_resolution(s):
        nums = re.findall(r'\d+', str(s))
        return (float(nums[0]), float(nums[1])) if len(nums) >= 2 else (np.nan, np.nan)
    
    df['Screen_Width'], df['Screen_Height'] = zip(*df['Screen'].apply(extract_resolution))
    
    # Fill missing values
    df['Screen_Width'] = df['Screen_Width'].fillna(df['Screen_Width'].median())
    df['Screen_Height'] = df['Screen_Height'].fillna(df['Screen_Height'].median())
    
    return df

# 2. Apply feature engineering to FULL dataset
df = create_features(df)

# 3. Define features AFTER feature engineering
numeric_features = ['RAM', 'Storage', 'Screen_Width', 'Screen_Height']
categorical_features = ['Manufacturer', 'CPU', 'GPU', 'Operating System']

# 4. Split into train/test (now with all features)
X = df[numeric_features + categorical_features]
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Verify all features exist
print("X_train columns:", X_train.columns.tolist())
assert all(col in X_train.columns for col in numeric_features + categorical_features)

# 6. Create and train pipeline
pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])),
    ('model', RandomForestRegressor(random_state=42))
])

pipeline.fit(X_train, y_train)

X_train columns: ['RAM', 'Storage', 'Screen_Width', 'Screen_Height', 'Manufacturer', 'CPU', 'GPU', 'Operating System']


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Making predictions
y_pred = pipeline.predict(X_test)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)**0.5  # Manual square root for RMSE
r2 = r2_score(y_test, y_pred)

# Print results
print(f'Mean Absolute Error (MAE): {mae:.2f}')
print(f'Mean Squared Error (MSE): {mse:.2f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.2f}') 
print(f'R-squared (R²): {r2:.2f}')

Mean Absolute Error (MAE): 1948501.41
Mean Squared Error (MSE): 8120857732231.86
Root Mean Squared Error (RMSE): 2849711.87
R-squared (R²): 0.67


In [22]:
# Calculate percentage within 10% of actual price
accuracy = (abs(y_pred - y_test) / y_test < 0.1).mean()
print(f'Percentage within 10% of actual price: {accuracy:.1%}')

Percentage within 10% of actual price: 36.9%


In [23]:
# For tighter tolerenace
# Check tighter tolerance (e.g., 5%)
(abs(y_pred - y_test) / y_test < 0.05).mean()

0.27692307692307694

In [26]:
# Data transformation

# Applying log to prices before training
y_train_log = np.log1p(y_train)

# Training model, then converting back predictions
pipeline.fit(X_train, y_train_log)
y_pred = np.expm1(pipeline.predict(X_test))  # Convert back

In [27]:
# Stratified Analysis

# Checking performance by price tiers

# Getting data-driven bins (5 bins with equal sample counts)
price_bins = np.percentile(y_test, [0, 20, 40, 60, 80, 100])
price_bins[0] = 0  # Force start at 0

print("Price Bin Analysis (MAE by Tier)")
print("="*40)
for i in range(len(price_bins)-1):
    mask = (y_test >= price_bins[i]) & (y_test < price_bins[i+1])
    n_samples = sum(mask)
    
    if n_samples > 0:
        mae = mean_absolute_error(y_test[mask], y_pred[mask])
        print(f"${price_bins[i]:,.0f}-${price_bins[i+1]:,.0f}:")
        print(f"  MAE: ${mae:,.2f} | Samples: {n_samples}")
    else:
        print(f"${price_bins[i]:,.0f}-${price_bins[i+1]:,.0f}: No samples")

Price Bin Analysis (MAE by Tier)
$0-$5,632,193:
  MAE: $776,608.88 | Samples: 13
$5,632,193-$7,936,999:
  MAE: $1,342,210.04 | Samples: 13
$7,936,999-$10,593,929:
  MAE: $1,629,831.61 | Samples: 13
$10,593,929-$13,789,714:
  MAE: $2,184,015.68 | Samples: 13
$13,789,714-$20,442,708:
  MAE: $2,728,674.92 | Samples: 12


In [28]:
# Save column names right after training

import json
with open('feature_columns.json', 'w') as f:
    json.dump(list(X_train.columns), f)  # Save the order of training columns

In [29]:
import joblib
# Save the entire pipeline (preprocessing + model)
joblib.dump(pipeline, 'laptop_price_predictor.pkl') 

['laptop_price_predictor.pkl']

In [30]:
# Loading the pipeline
pipeline = joblib.load('laptop_price_predictor.pkl')
print("Pipeline loaded successfully! Contains:", [step[0] for step in pipeline.steps])

Pipeline loaded successfully! Contains: ['preprocessor', 'model']


In [31]:
# Example of usage:

import pandas as pd
import joblib
import json

# 1. Load the pipeline and feature columns
pipeline = joblib.load('laptop_price_predictor.pkl')
with open('feature_columns.json', 'r') as f:
    train_columns = json.load(f)

# 2. Prepare new data (with ALL original features)
new_data = pd.DataFrame({
    'Manufacturer': ['Dell'],
    'RAM': [16],
    'Storage': [512],
    'Screen': ['1920x1080'],  # Original screen column
    'CPU': ['Intel i7'],
    'GPU': ['NVIDIA RTX 3060'],
    'Operating System': ['Windows']
})

# 3. Generate derived features (same as during training)
new_data['Screen_Width'] = new_data['Screen'].str.split('x').str[0].astype(float)
new_data['Screen_Height'] = new_data['Screen'].str.split('x').str[1].astype(float)

# 4. Reorder columns to match training exactly
new_data = new_data[train_columns]

# 5. Predict
try:
    predicted_price = pipeline.predict(new_data)[0]
    print(f"Predicted Price: ${predicted_price:,.2f}")
except Exception as e:
    print("Error during prediction:", str(e))


Predicted Price: $16.44


In [32]:
print("Original price range:")
print(f"Min: ${y_train.min():,.2f}")
print(f"Max: ${y_train.max():,.2f}")
print(f"Median: ${y_train.median():,.2f}")

Original price range:
Min: $1,547,208.00
Max: $35,345,700.00
Median: $8,878,662.00


In [33]:
print("Pipeline steps:")
print(pipeline.steps)

Pipeline steps:
[('preprocessor', ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['RAM', 'Storage', 'Screen_Width',
                                  'Screen_Height']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['Manufacturer', 'CPU', 'GPU',
                                  'Operating System'])])), ('model', RandomForestRegressor(random_state=42))]


In [34]:
# Predict on training data (should give realistic values)
sample_pred = pipeline.predict(X_train[:1])[0]
print(f"Predicted price for training sample: ${sample_pred:,.2f}")
print(f"Actual price: ${y_train.iloc[0]:,.2f}")

Predicted price for training sample: $14.98
Actual price: $3,014,388.00
